# Notebook Air-Quality Beijing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
import sys
from sklearn.preprocessing import StandardScaler # Mantener para uso directo/verificación potencial
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import SmoothL1Loss

# Asegurar que la raíz del proyecto esté en el Python path
# Ajusta la profundidad del path ('../') según sea necesario
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Importar módulos refactorizados
from pampaneira_imputation import config
from pampaneira_imputation import data_preprocessor as dp
from pampaneira_imputation import imputation_methods as im
from pampaneira_imputation import evaluation as ev
from pampaneira_imputation import utils


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 



In [3]:
from benchpots.datasets import preprocess_beijing_air_quality

# Load the dataset with artificially missing values
beijing_air_quality = preprocess_beijing_air_quality(subset="all", rate=0.1, n_steps=24)

2025-06-04 18:40:18 [INFO]: You're using dataset beijing_multisite_air_quality, please cite it properly in your work. You can find its reference information at the below link: 
https://github.com/WenjieDu/TSDB/tree/main/dataset_profiles/beijing_multisite_air_quality
2025-06-04 18:40:18 [INFO]: Dataset beijing_multisite_air_quality has already been downloaded. Processing directly...
2025-06-04 18:40:18 [INFO]: Dataset beijing_multisite_air_quality has already been cached. Loading from cache directly...
2025-06-04 18:40:18 [INFO]: Loaded successfully!
2025-06-04 18:40:18 [INFO]: Current dataframe shape: (35064, 18)
/Users/eigenric/Library/CloudStorage/Dropbox/dgiim/TFG/.venv/lib/python3.13/site-packages/benchpots/datasets/beijing_multisite_air_quality.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/u

In [4]:
beijing_air_quality.keys()

dict_keys(['n_steps', 'n_features', 'scaler', 'train_X', 'val_X', 'test_X', 'train_X_ori', 'val_X_ori', 'test_X_ori'])

In [5]:
# Assemble the datasets for training, validating, and testing.

dataset_for_training = {
    "X": beijing_air_quality['train_X'],
}

dataset_for_validating = {
    "X": beijing_air_quality['val_X'],
    "X_ori": beijing_air_quality['val_X_ori'],
}

dataset_for_testing = {
    "X": beijing_air_quality['test_X'],
    "X_ori": beijing_air_quality['test_X_ori']
}


## División en entrenamiento, validación y testing

 The test set
takes data from the first 10 months (2013/03 - 2013/12). The validation set contains data from the following 10 months
(2014/01 - 2014/10). The training set takes the left 28 months (2014/11 - 2017/02). T

In [6]:
# Aplicar métodos base a los datos del conjunto de test (`dataset_for_testing['X']`)
X_test_missing = dataset_for_testing['X']
imputed_results = {} # Diccionario para almacenar resultados

print("\nEjecutando métodos de imputación base...")


Ejecutando métodos de imputación base...


#### 4.1 Imputación por Mediana

Rellenamos los valores faltantes utilizando la mediana de cada característica (columna) calculada sobre todo el conjunto de datos de test (a través de todas las muestras y pasos de tiempo).

In [7]:
imputed_results['median'] = im.impute_median_sample_wise(X_test_missing)
print(f"Forma imputación mediana: {imputed_results['median'].shape}, NaNs: {np.isnan(imputed_results['median']).sum()}")

  Midiendo tiempo para 'impute_median_sample_wise'...
  Finalizado 'impute_median_sample_wise' en 1.447903 segundos
Forma imputación mediana: (304, 24, 132), NaNs: 0


#### 4.2 Imputación por Media

Similar a la mediana, pero usamos la media de cada característica para rellenar los valores faltantes.

In [8]:
imputed_results['mean'] = im.impute_mean_sample_wise(X_test_missing)

print(f"Forma imputación media: {imputed_results['mean'].shape}, NaNs: {np.isnan(imputed_results['mean']).sum()}")

  Midiendo tiempo para 'impute_mean_sample_wise'...
  Finalizado 'impute_mean_sample_wise' en 0.885563 segundos
Forma imputación media: (304, 24, 132), NaNs: 0


#### 4.4 Relleno Forward (ffill)

Nota: Estos métodos pueden requerir eliminar columnas como `WS`, `WD` si causan problemas o no son necesarias. La función de imputación en sí no elimina columnas, pero la evaluación podría necesitar alineación. Asumiremos por ahora que imputamos todas las características. La evaluación maneja la posible eliminación de columnas. Alternativamente, modificamos los datos de entrada antes de pasarlos a la imputación:

In [9]:
# ### 4.4 Relleno Forward/Backward (ffill/bfill)
# Nota: Estos métodos pueden requerir eliminar columnas como WS, WD si causan problemas.
# Modificamos los datos de entrada si es necesario:
X_test_missing_no_ws_wd = X_test_missing.copy()
cols_to_drop_indices = [config.FEATURE_COLUMNS.index(col) for col in config.COLS_TO_DROP_FOR_BASELINE if col in config.FEATURE_COLUMNS]

if cols_to_drop_indices:
    print(f"Eliminando columnas en índices {cols_to_drop_indices} para métodos Fill.")
    X_test_missing_no_ws_wd = np.delete(X_test_missing, cols_to_drop_indices, axis=2)
else:
    X_test_missing_no_ws_wd = X_test_missing # No hay columnas que eliminar

imputed_results['ffill'] = im.impute_forward(X_test_missing_no_ws_wd)
print(f"Forma imputación Forward fill {imputed_results['ffill'].shape}, NaNs: {np.isnan(imputed_results['ffill']).sum()}")

Eliminando columnas en índices [111, 112] para métodos Fill.
  Midiendo tiempo para 'impute_forward'...
  Finalizado 'impute_forward' en 0.075557 segundos
Forma imputación Forward fill (304, 24, 130), NaNs: 0


### 4.5. Modelo Transformer

In [10]:
transformer_model = im.fit_transformer(dataset_for_training, dataset_for_validating)
imputed_results['transformer'] = im.impute_transformer(transformer_model,
                                                       dataset_for_testing)

2025-06-04 18:40:22 [INFO]: No given device, using default device: cpu
2025-06-04 18:40:22 [INFO]: Model files will be saved to ../results/imputation/transformer/20250604_T184022
2025-06-04 18:40:22 [INFO]: Tensorboard file will be saved to ../results/imputation/transformer/20250604_T184022/tensorboard
2025-06-04 18:40:22 [INFO]: Using customized MAE as the training loss function.
2025-06-04 18:40:22 [INFO]: Using customized MSE as the validation metric function.
2025-06-04 18:40:22 [INFO]: Transformer initialized with the given hyperparameters, the number of trainable parameters: 446,852


  Midiendo tiempo para 'fit_transformer'...

---> Ejecutando lógica de imputación con Transformer...
    Inicializando Transformer model...
    Configurando optimizer, scheduler, loss...
    Training Transformer model...


/Users/eigenric/Library/CloudStorage/Dropbox/dgiim/TFG/.venv/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
2025-06-04 18:40:23 [INFO]: Epoch 001 - training loss (MAE): 1.2463, validation MSE: 0.5877
2025-06-04 18:40:25 [INFO]: Epoch 002 - training loss (MAE): 0.8694, validation MSE: 0.4431
2025-06-04 18:40:27 [INFO]: Epoch 003 - training loss (MAE): 0.7422, validation MSE: 0.3866
2025-06-04 18:40:28 [INFO]: Epoch 004 - training loss (MAE): 0.6875, validation MSE: 0.3587
2025-06-04 18:40:30 [INFO]: Epoch 005 - training loss (MAE): 0.6420, validation MSE: 0.3336
2025-06-04 18:40:31 [INFO]: Epoch 006 - training loss (MAE): 0.6006, validation MSE: 0.3128
2025-06-04 18:40:33 [INFO]: Epoch 007 - training loss (MAE): 0.5764, validation MSE: 0.3000
2025-06-04 18:40:34 [INFO]: Epoch 008 - training loss (MAE): 0.5543, validation MSE: 0.2912
2025-06-04 18:40:36 [

    Transformer training complete.
  Finalizado 'fit_transformer' en 51.986739 segundos
  Midiendo tiempo para 'impute_transformer'...
    Imputando con Transformer en el conjunto de test...
    Imputación Transformer completa.
  Finalizado 'impute_transformer' en 0.211625 segundos


### 4.6. Modelo SAITS

Si la biblioteca `pypots` está disponible, configuramos y entrenamos un modelo SAITS (Self-Attention based Imputation for Time Series) usando los datos de entrenamiento y validación preprocesados. Luego, usamos el modelo entrenado para imputar los valores faltantes en el conjunto de test.

In [11]:
saits_model = im.fit_saits(dataset_for_training, dataset_for_validating)
imputed_results['saits'] = im.impute_saits(saits_model,
                                           dataset_for_testing)

2025-06-04 18:41:14 [INFO]: No given device, using default device: cpu
2025-06-04 18:41:14 [INFO]: Model files will be saved to ../results/imputation/saits/20250604_T184114
2025-06-04 18:41:14 [INFO]: Tensorboard file will be saved to ../results/imputation/saits/20250604_T184114/tensorboard
2025-06-04 18:41:14 [INFO]: Using customized MAE as the training loss function.
2025-06-04 18:41:14 [INFO]: Using customized MSE as the validation metric function.
2025-06-04 18:41:14 [INFO]: SAITS initialized with the given hyperparameters, the number of trainable parameters: 931,984


  Midiendo tiempo para 'fit_saits'...

Configurando y ejecutando imputación con SAITS...
Entrenando modelo SAITS...


2025-06-04 18:41:16 [INFO]: Epoch 001 - training loss (MAE): 1.1929, validation MSE: 0.5697
2025-06-04 18:41:19 [INFO]: Epoch 002 - training loss (MAE): 0.8277, validation MSE: 0.4251
2025-06-04 18:41:22 [INFO]: Epoch 003 - training loss (MAE): 0.7046, validation MSE: 0.3682
2025-06-04 18:41:24 [INFO]: Epoch 004 - training loss (MAE): 0.6463, validation MSE: 0.3382
2025-06-04 18:41:27 [INFO]: Epoch 005 - training loss (MAE): 0.6002, validation MSE: 0.3173
2025-06-04 18:41:32 [INFO]: Epoch 006 - training loss (MAE): 0.5702, validation MSE: 0.2977
2025-06-04 18:41:38 [INFO]: Epoch 007 - training loss (MAE): 0.5402, validation MSE: 0.2817
2025-06-04 18:41:41 [INFO]: Epoch 008 - training loss (MAE): 0.5187, validation MSE: 0.2746
2025-06-04 18:41:43 [INFO]: Epoch 009 - training loss (MAE): 0.5037, validation MSE: 0.2681
2025-06-04 18:41:48 [INFO]: Epoch 010 - training loss (MAE): 0.4952, validation MSE: 0.2592
2025-06-04 18:41:51 [INFO]: Epoch 011 - training loss (MAE): 0.4827, validation 

Entrenamiento de SAITS completo.
  Finalizado 'fit_saits' en 138.286403 segundos
  Midiendo tiempo para 'impute_saits'...
Imputando con SAITS en el conjunto de test...
  Finalizado 'impute_saits' en 0.394236 segundos


In [12]:
processed_data = beijing_air_quality

In [13]:
 # a) Máscara de NaNs PREEXISTENTES (antes de la introducción artificial)
train_preexisting_nan_mask = np.isnan(processed_data["train_X_ori"])
val_preexisting_nan_mask = np.isnan(processed_data["val_X_ori"])
test_preexisting_nan_mask = np.isnan(processed_data["test_X_ori"])
processed_data["train_preexisting_nan_mask"] = train_preexisting_nan_mask.astype(int)
processed_data["val_preexisting_nan_mask"] = val_preexisting_nan_mask.astype(int)
processed_data["test_preexisting_nan_mask"] = test_preexisting_nan_mask.astype(int)

# Actualiza los datos en el diccionario
train_X_missing = processed_data["train_X"] 
val_X_missing = processed_data["val_X"] 
test_X_missing = processed_data["test_X"] 

# c) Máscara indicadora TOTAL (NaNs originales + artificiales)
train_indicating_mask = np.isnan(train_X_missing)
val_indicating_mask = np.isnan(val_X_missing)
test_indicating_mask = np.isnan(test_X_missing)
processed_data["train_indicating_mask"] = train_indicating_mask.astype(int)
processed_data["val_indicating_mask"] = val_indicating_mask.astype(int)
processed_data["test_indicating_mask"] = test_indicating_mask.astype(int)

# d) Máscara de NaNs ARTIFICIALES (SOLO los introducidos ahora)
# Es True donde NO era NaN antes (~preexisting) Y SÍ es NaN ahora (indicating)
train_artificial_nan_mask = ~train_preexisting_nan_mask & train_indicating_mask
val_artificial_nan_mask = ~val_preexisting_nan_mask & val_indicating_mask
test_artificial_nan_mask = ~test_preexisting_nan_mask & test_indicating_mask
processed_data["train_artificial_mask"] = train_artificial_nan_mask.astype(int)
processed_data["val_artificial_mask"] = val_artificial_nan_mask.astype(int)
processed_data["test_artificial_mask"] = test_artificial_nan_mask.astype(int)

In [14]:
print("\nEvaluando rendimiento de imputación en el conjunto de test...")
import pampaneira_imputation.evaluation as ev

# Usar la función del módulo de evaluación
# Requiere el diccionario de datos preprocesados (para ground truth y máscara)
# y el diccionario de resultados imputados
evaluation_table = ev.evaluate_all_methods(
    beijing_air_quality,
    imputed_results
)

evaluation_table.T


Evaluando rendimiento de imputación en el conjunto de test...

Calculando métricas para: median

Calculando métricas para: mean
Advertencia: No se encontraron resultados imputados para el método 'linear'. Saltando.
Ajustando datos verdaderos y máscara para ffill debido a columnas eliminadas para ffill.

Calculando métricas para: ffill
Advertencia: No se encontraron resultados imputados para el método 'bfill'. Saltando.

Calculando métricas para: transformer

Calculando métricas para: saits


Method,Median,Mean,Ffill,Transformer,Saits
RMSE,0.6435,0.6170,0.5493,0.5320,0.5156
MSE,0.4141,0.3806,0.3018,0.2831,0.2659
MAE,0.3116,0.3202,0.1896,0.1940,0.1793
MRE,0.4145,0.4259,0.2517,0.2580,0.2385
